# Step 1 - Initial Dataset processing

```sql
CREATE TABLE "AllGalaxies"
(
    "OriginalDatasetId" varchar(100) not null,
    "OriginalDataset" varchar(100) not null,
    "DebiasedSpiralProbability" double precision not null,
    "DebiasedSpiralFlag" boolean not null,
    "DebiasedEllipticalProbability" double precision not null,
    "DebiasedEllipticalFlag" boolean not null,
    "ra" double precision, -- nullable to allow to update GZ1 coords later
    "dec" double precision
);
```

## GZ1

In [1]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

def map_ra(ra, dec):
    coords = SkyCoord(ra, dec, unit=(unit.deg, unit.deg)).transform_to("fk5")
    return coords.ra.deg

gz1_data = pd.read_csv("GZ1/GalaxyZoo1_DR_table2.csv")

# there is no data cleaning for GZ1 done
gz1_clean = gz1_data

gz1_clean["OriginalDataset"] = "GZ1"
gz1_clean = gz1_clean.rename(columns={
    "P_EL_DEBIASED" : "DebiasedEllipticalProbability",
    "P_CS_DEBIASED" : "DebiasedSpiralProbability",
    "OBJID": "OriginalDatasetId"
    }).reset_index(drop=True)

gz1_clean["DebiasedSpiralFlag"] = gz1_clean.apply(lambda r: 1 if r["SPIRAL"] == 1 else 0, axis = 1)
gz1_clean["DebiasedEllipticalFlag"] = gz1_clean.apply(lambda r: 1 if r["ELLIPTICAL"] == 1 else 0, axis = 1)

gz1_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedSpiralFlag", "DebiasedEllipticalProbability", "DebiasedEllipticalFlag", "OriginalDataset"]].to_csv("GZ1/GZ1_unfiltered.csv", index=False)

### CasJobs query to get standard ra and dec

```sql
SELECT g.originaldatasetid, s.ra, s.dec
into mydb.GZ1_ALL_Coords
from MyDB.GZ1_ALL AS g
JOIN DR7.PhotoObj AS s
ON s.objID = g.originaldatasetid
```

### query to update ra and dec of GZ1 galaxies from results above

```sql
UPDATE "AllGalaxies" AS g
SET ra = c.ra, dec = c.dec
FROM "GZ1_uinfiltered_coords" AS c
WHERE g."OriginalDataset" = 'GZ1'
AND g."OriginalDatasetId" = c.originaldatasetid;
```

## GZ2

In [ ]:
gz2_data = pd.read_csv("GZ2/gz2_hart16.csv")

# remove all artefacts
gz2_clean = gz2_data[gz2_data["gz2_class"] != 'A']

gz2_clean["OriginalDataset"] = "GZ2"

gz2_clean = gz2_clean.rename(columns={
    "t01_smooth_or_features_a02_features_or_disk_flag" : "DebiasedSpiralFlag",
    "t01_smooth_or_features_a02_features_or_disk_debiased" : "DebiasedSpiralProbability",
    "t01_smooth_or_features_a01_smooth_flag" : "DebiasedEllipticalFlag",
    "t01_smooth_or_features_a01_smooth_debiased" : "DebiasedEllipticalProbability",
    "dr7objid": "OriginalDatasetId"
    }).reset_index(drop=True)

gz2_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedSpiralFlag", "DebiasedEllipticalProbability", "DebiasedEllipticalFlag", "OriginalDataset", "ra", "dec"]].to_csv("GZ2/GZ2_unfiltered.csv", index= False)


## GZ Hubble

In [ ]:
gzh_data = pd.read_csv("GZH/gz_hubble_main.csv")

# remove all artefacts
gzh_clean = gzh_data[gzh_data["t01_smooth_or_features_a03_star_or_artifact_flag"] != 1]

gzh_clean["OriginalDataset"] = "GZ_HUBBLE"

gzh_clean = gzh_clean.rename(columns={
    "t01_smooth_or_features_a02_features_or_disk_flag" : "DebiasedSpiralFlag",
    "t01_smooth_or_features_a02_features_or_disk_debiased_fraction" : "DebiasedSpiralProbability",
    "t01_smooth_or_features_a01_smooth_flag" : "DebiasedEllipticalFlag",
    "t01_smooth_or_features_a01_smooth_debiased_fraction" : "DebiasedEllipticalProbability",
    "zooniverse_id": "OriginalDatasetId",
    "RA": "ra",
    "DEC": "dec"
    }).reset_index(drop=True)

gzh_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedSpiralFlag", "DebiasedEllipticalProbability", "DebiasedEllipticalFlag", "OriginalDataset", "ra", "dec"]].to_csv("GZH/GZH_unfiltered.csv", index= False)

## GZ DECALS

In [ ]:
gzdec_1_2_data = pd.read_csv("GZD/gz_decals_volunteers_1_and_2.csv")

# # remove all artefacts
gzd_clean = gzdec_1_2_data[gzdec_1_2_data["smooth-or-featured_artifact_debiased"] < 0.5]

gzd_clean["OriginalDataset"] = "GZ_DECALS_12"

gzd_clean["DebiasedSpiralFlag"] = gzd_clean["smooth-or-featured_featured-or-disk_debiased"] >= 0.5
gzd_clean["DebiasedEllipticalFlag"] = gzd_clean["smooth-or-featured_smooth_debiased"] >= 0.5

gzd_clean = gzd_clean.rename(columns={
    "smooth-or-featured_featured-or-disk_debiased" : "DebiasedSpiralProbability",
    "smooth-or-featured_smooth_debiased" : "DebiasedEllipticalProbability",
    "iauname": "OriginalDatasetId",
    }).reset_index(drop=True)

gzd_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedSpiralFlag", "DebiasedEllipticalProbability", "DebiasedEllipticalFlag", "OriginalDataset", "ra", "dec"]].to_csv("GZD/GZD_12_unfiltered.csv", index= False)

In [ ]:
gzdec_5_data = pd.read_csv("GZD/gz_decals_volunteers_5.csv")

# # remove all artefacts
gzd_5_clean = gzdec_5_data[gzdec_5_data["smooth-or-featured_artifact_debiased"] < 0.5]

gzd_5_clean["OriginalDataset"] = "GZ_DECALS_5"

gzd_5_clean["DebiasedSpiralFlag"] = gzd_5_clean["smooth-or-featured_featured-or-disk_debiased"] >= 0.5
gzd_5_clean["DebiasedEllipticalFlag"] = gzd_5_clean["smooth-or-featured_smooth_debiased"] >= 0.5

gzd_5_clean = gzd_5_clean.rename(columns={
    "smooth-or-featured_featured-or-disk_debiased" : "DebiasedSpiralProbability",
    "smooth-or-featured_smooth_debiased" : "DebiasedEllipticalProbability",
    "iauname": "OriginalDatasetId",
    }).reset_index(drop=True)

gzd_5_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedSpiralFlag", "DebiasedEllipticalProbability", "DebiasedEllipticalFlag", "OriginalDataset", "ra", "dec"]].to_csv("GZD/GZD_5_unfiltered.csv", index= False)

## GZ CANDELS

In [ ]:
gzc_data = pd.read_csv("GZC/gz_candels_table_2_main_release.csv")

# # remove all artefacts
gzc_clean = gzc_data[gzc_data["t00_smooth_or_featured_a2_artifact_weighted_frac"] < 0.5]

gzc_clean["OriginalDataset"] = "GZ_CANDELS"

gzc_clean["DebiasedSpiralFlag"] = gzc_clean["t00_smooth_or_featured_a1_features_weighted_frac"] >= 0.5
gzc_clean["DebiasedEllipticalFlag"] = gzc_clean["t00_smooth_or_featured_a0_smooth_weighted_frac"] >= 0.5

gzc_clean = gzc_clean.rename(columns={
    "t00_smooth_or_featured_a1_features_weighted_frac" : "DebiasedSpiralProbability",
    "t00_smooth_or_featured_a0_smooth_weighted_frac" : "DebiasedEllipticalProbability",
    "ID": "OriginalDatasetId",
    "RA": "ra",
    "Dec": "dec"
    }).reset_index(drop=True)

gzc_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedSpiralFlag", "DebiasedEllipticalProbability", "DebiasedEllipticalFlag", "OriginalDataset", "ra", "dec"]].to_csv("GZC/GZC_unfiltered.csv", index= False)

---
# Step 2 - Data cleanup, analysis and merging by original dataset ID

## Assign a class

```sql
ALTER TABLE "AllGalaxies"
ADD COLUMN "BinaryClass" text;

UPDATE "AllGalaxies"
SET "BinaryClass" =
    CASE
       WHEN "DebiasedSpiralFlag" AND NOT "DebiasedEllipticalFlag" THEN 'Spiral'
       WHEN "DebiasedEllipticalFlag" AND NOT "DebiasedSpiralFlag" THEN 'Elliptical'
       ELSE 'Uncertain'
    END;
```

## Filter only certain galaxies

```sql
SELECT *
INTO "AllCertainGalaxies"
FROM "AllGalaxies"
WHERE "BinaryClass" != 'Uncertain';

CREATE INDEX IF NOT EXISTS datasetID ON "AllCertainGalaxies"("OriginalDatasetId");
CREATE INDEX IF NOT EXISTS dataset ON "AllCertainGalaxies"("OriginalDataset");
CREATE UNIQUE INDEX IF NOT EXISTS id_per_dataset ON "AllCertainGalaxies"("OriginalDatasetId", "OriginalDataset");
```

## Merge galaxies by original dataset ID

```sql
WITH gz1 AS (
    SELECT * FROM "AllCertainGalaxies"
    WHERE "OriginalDataset" = 'GZ1'
),
gz2 AS (
    SELECT * FROM "AllCertainGalaxies"
    WHERE "OriginalDataset" = 'GZ2'
),
gzh AS (
    SELECT * FROM "AllCertainGalaxies"
    WHERE "OriginalDataset" = 'GZ_HUBBLE'
),
gzd AS (
    SELECT * FROM "AllCertainGalaxies"
    WHERE "OriginalDataset" = 'GZ_DECALS_12' OR "OriginalDataset" = 'GZ_DECALS_5'
),
gzc AS (
    SELECT * FROM "AllCertainGalaxies"
    WHERE "OriginalDataset" = 'GZ_CANDELS'
)
SELECT
    gen_random_uuid() AS "Id",
    main.*,
    gz1."OriginalDatasetId" AS "GZ1_Id",
    gz1."BinaryClass" AS "GZ1_BinaryClass",
    gz2."OriginalDatasetId" AS "GZ2_Id",
    gz2."BinaryClass" AS "GZ2_BinaryClass",
    gzh."OriginalDatasetId" AS "GZH_Id",
    gzh."BinaryClass" AS "GZH_BinaryClass",
    gzd."OriginalDatasetId" AS "GZD_Id",
    gzd."BinaryClass" AS "GZD_BinaryClass",
    gzc."OriginalDatasetId" AS "GZC_Id",
    gzc."BinaryClass" AS "GZC_BinaryClass",
    CASE WHEN
        (
            WITH z as (SELECT DISTINCT UNNEST(array[gz1."BinaryClass", gz2."BinaryClass", gzh."BinaryClass", gzd."BinaryClass", gzc."BinaryClass"]) AS "Class")
            SELECT COUNT(*) FROM z
            WHERE z."Class" IS NOT NULL
        ) = 1
        THEN true
        ELSE false END AS "AllClassesMatch"
INTO "MatchedGalaxiesById"
FROM "AllCertainGalaxies" AS main
LEFT JOIN gz1
ON main."OriginalDatasetId" = gz1."OriginalDatasetId"
LEFT JOIN gz2
ON main."OriginalDatasetId" = gz2."OriginalDatasetId"
LEFT JOIN gzh
ON main."OriginalDatasetId" = gzh."OriginalDatasetId"
LEFT JOIN gzd
ON main."OriginalDatasetId" = gzd."OriginalDatasetId"
LEFT JOIN gzc
ON main."OriginalDatasetId" = gzc."OriginalDatasetId"
```

```sql
DELETE FROM "MatchedGalaxiesById"
WHERE "AllClassesMatch" = false
```

```sql
DELETE
FROM "MatchedGalaxiesById"
WHERE ctid IN (
        SELECT ctid
        FROM (
            SELECT ctid, row_number()
            OVER (PARTITION BY "OriginalDatasetId" ORDER BY ctid) rn
            FROM "MatchedGalaxiesById"
        ) t
        WHERE rn > 1
        );
```

# Step 3 - Gather z and mag

```sql
ALTER TABLE "MatchedGalaxiesById"
    ADD COLUMN "z" double precision,
    ADD COLUMN "z_type" text;
```

## GZ1 and GZ2

Cas Job query
```sql
SELECT g.Id, g.OriginalDatasetId, s.z AS "Spec_z", s.zErr AS "Spec_z_err", sp.petroMag_r as "Spec_mag", sp.petroMagErr_r AS "Spec_mag_err"
FROM MyDB.MatchedGalaxiesById_GZ1_GZ2 AS g
  LEFT JOIN DR7.SpecObj AS s
  ON s.bestObjId = g.OriginalDatasetId
  AND s.objTypeName = 'GALAXY'
  LEFT JOIN DR7.SpecPhoto AS sp
  ON sp.objID = g.OriginalDatasetId
```

Import original csv file as GZ1_GZ2_z_and_mag_data table, preserving all values as float8
```sql
UPDATE "MatchedGalaxiesById" AS o
SET "z" = d."spec_z", "z_type" = 'spec'
FROM data."gz1_gz2_z_and_mag_data" AS d
WHERE "OriginalDataset" IN ('GZ1', 'GZ2')
  AND d.originaldatasetid = o."OriginalDatasetId"
  AND d."spec_z" IS NOT NULL;
```

## GZ Decals

Import original csv files as gz_decals_volunteers_1_and_2 and gz_decals_volunteers_5, preserving `redshift` as float8
```sql
UPDATE "MatchedGalaxiesById" AS o
SET "z" = d."redshift", "z_type" = 'original_gzd'
FROM data.gz_decals_volunteers_1_and_2 AS d
WHERE "OriginalDataset" = 'GZ_DECALS_12' 
    AND d."iauname" = o."OriginalDatasetId";

UPDATE "MatchedGalaxiesById" AS o
SET "z" = d."redshift", "z_type" = 'original_gzd'
FROM data.gz_decals_volunteers_5 AS d
WHERE "OriginalDataset" = 'GZ_DECALS_5'
  AND d."iauname" = o."OriginalDatasetId";
```

## GZ Hubble

```sql
UPDATE "MatchedGalaxiesById" AS o
SET "z" = d."z_best", z_type = 'original_gzh'
FROM data.gz_hubble_main AS d
WHERE "OriginalDataset" = 'GZ_HUBBLE'
  AND d."zooniverse_id" = o."OriginalDatasetId"
  AND z_best != 999;
```

## GZ Candels

In [7]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

# file source: https://archive.stsci.edu/hlsp/candels/cosmos-catalogs
candels_redshifts = pd.read_csv("GZC/hlsp_candels_hst_wfc3_cosmos_multi_v2_redshift-cat.txt", comment='#', sep=' ', header=None, usecols=range(1,8), names =[
"ID",
"RA",
"DEC",
"z_best",
"z_best_type",
"z_spec",
"z_spec_ref"
]).to_csv("GZC/gz_candels_cosmos_redshifts.csv", index=False)

```sql
UPDATE "MatchedGalaxiesById" AS o
SET "z" = d."z_best", "z_type" = CASE WHEN d.z_best_type = 'p' THEN 'photo' ELSE 'spec' END
FROM data.gz_candels_cosmos_redshifts AS d
WHERE "OriginalDataset" = 'GZ_CANDELS' 
    AND concat('COS_', d."id") = o."OriginalDatasetId";
```

In [9]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

# file source: https://archive.stsci.edu/hlsp/candels/goods-s-catalogs
candels_redshifts = pd.read_csv("GZC/hlsp_candels_hst_wfc3_goodss_multi_v2_redshift-cat.txt", comment='#', sep=' ', header=None, usecols=range(1,8), names =[
"ID",
"RA",
"DEC",
"z_best",
"z_best_type",
"z_spec",
"z_spec_ref"
])

candels_redshifts.to_csv("GZC/gz_candels_gooods_redshifts.csv", index=False)

```sql
UPDATE "MatchedGalaxiesById" AS o
SET "z" = d."z_best", "z_type" = CASE WHEN d.z_best_type = 'p' THEN 'photo' ELSE 'spec' END
FROM data.gz_candels_gooods_redshifts AS d
WHERE "OriginalDataset" = 'GZ_CANDELS'
    AND concat('GDS_', d."id") = o."OriginalDatasetId";
```

In [10]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

# file source: https://archive.stsci.edu/hlsp/candels/uds-catalogs
candels_redshifts = pd.read_csv("GZC/hlsp_candels_hst_wfc3_uds_multi_v2_redshift-cat.txt", comment='#', sep=' ', header=None, usecols=range(1,8), names =[
"ID",
"RA",
"DEC",
"z_best",
"z_best_type",
"z_spec",
"z_spec_ref"
])

candels_redshifts.to_csv("GZC/gz_candels_uds_redshifts.csv", index=False)

```sql
UPDATE "MatchedGalaxiesById" AS o
SET "z" = d."z_best", "z_type" = CASE WHEN d.z_best_type = 'p' THEN 'photo' ELSE 'spec' END
FROM data.gz_candels_uds_redshifts AS d
WHERE "OriginalDataset" = 'GZ_CANDELS'
    AND concat('UDS_', d."id") = o."OriginalDatasetId";
```

# Step 4 - Merge by coordinates

Create crossmatch columns:
```sql
ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZ12_Id" text;

ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZ12_Class" text;

ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZD_Id" text;

ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZD_Class" text;

ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZH_Id" text;

ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZH_Class" text;

ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZC_Id" text;

ALTER TABLE "MatchedGalaxiesById"
ADD COLUMN "Coordinates_GZC_Class" text;
```

Add coordinates postgis column:

```sql
CREATE EXTENSION postgis;

SELECT *, st_point(ra, dec) AS "Coordinates"
INTO "MatchedGalaxiesWithCoordinates"
FROM "MatchedGalaxiesById";

-- spatial index
CREATE INDEX IDX_coordinates ON "MatchedGalaxiesWithCoordinates" USING GIST("Coordinates")
```


Cross-match by 0.5 arcsec distance
```sql
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZ12_Id" = null, "Coordinates_GZ12_Class" = null,
    "Coordinates_GZH_Id" = null, "Coordinates_GZH_Class" = null,
    "Coordinates_GZC_Id" = null, "Coordinates_GZC_Class" = null,
    "Coordinates_GZD_Id" = null, "Coordinates_GZD_Class" = null;

WITH gz12 As (
    SELECT "OriginalDatasetId", "BinaryClass", "Coordinates", ra, dec, "z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" IN ('GZ1', 'GZ2')
),
     sub AS (
         SELECT main."Id", gz12."OriginalDatasetId", gz12."BinaryClass"
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  LEFT JOIN gz12
                            ON st_dwithin(gz12."Coordinates", main."Coordinates", 0.000277778 / 2) -- 0.5 arcsec
         WHERE gz12."OriginalDatasetId" IS NOT NULL
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZ12_Id" = sub."OriginalDatasetId", "Coordinates_GZ12_Class" = sub."BinaryClass"
FROM sub
WHERE main."Id" = sub."Id";

 ---------------

WITH gzh As (
    SELECT "OriginalDatasetId", "BinaryClass", "Coordinates", ra, dec, "z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ_HUBBLE'
),
     sub AS (
         SELECT main."Id", gzh."OriginalDatasetId", gzh."BinaryClass"
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  LEFT JOIN gzh
                            ON st_dwithin(gzh."Coordinates", main."Coordinates", 0.000277778 / 2) -- 0.5 arcsec
         WHERE gzh."OriginalDatasetId" IS NOT NULL
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZH_Id" = sub."OriginalDatasetId", "Coordinates_GZH_Class" = sub."BinaryClass"
FROM sub
WHERE main."Id" = sub."Id";

 ---------------


WITH gzc As (
    SELECT "OriginalDatasetId", "BinaryClass", "Coordinates", ra, dec, "z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ_CANDELS'
),
     sub AS (
         SELECT main."Id", gzc."OriginalDatasetId", gzc."BinaryClass"
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  LEFT JOIN gzc
                            ON st_dwithin(gzc."Coordinates", main."Coordinates", 0.000277778 / 2) -- 0.5 arcsec
         WHERE gzc."OriginalDatasetId" IS NOT NULL
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZC_Id" = sub."OriginalDatasetId", "Coordinates_GZC_Class" = sub."BinaryClass"
FROM sub
WHERE main."Id" = sub."Id";


 ---------------

WITH gzd As (
    SELECT "OriginalDatasetId", "BinaryClass", "Coordinates", ra, dec, "z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" IN ('GZ_DECALS_12', 'GZ_DECALS_5')
),
     sub AS (
         SELECT main."Id", gzd."OriginalDatasetId", gzd."BinaryClass"
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  LEFT JOIN gzd
                            ON st_dwithin(gzd."Coordinates", main."Coordinates", 0.000277778 / 2) -- 0.5 arcsec
         WHERE gzd."OriginalDatasetId" IS NOT NULL
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZD_Id" = sub."OriginalDatasetId", "Coordinates_GZD_Class" = sub."BinaryClass"
FROM sub
WHERE main."Id" = sub."Id";
```

```sql
SELECT
    main.*,
    CASE WHEN
        (
            WITH z as (SELECT DISTINCT UNNEST(array["Coordinates_GZ12_Class", "Coordinates_GZH_Class", "Coordinates_GZC_Class", "Coordinates_GZD_Class"]) AS "Class")
            SELECT COUNT(*) FROM z
            WHERE z."Class" IS NOT NULL
        ) = 1
        THEN true
        ELSE false END AS "AllCoordinateClassesMatch"
INTO "FullyCrossMatchedGalaxies"
FROM "MatchedGalaxiesWithCoordinates" AS main

DELETE FROM "FullyCrossMatchedGalaxies"
WHERE "AllCoordinateClassesMatch" = false;
```

Delete duplicate galaxies:
```sql
DELETE
FROM "FullyCrossMatchedGalaxies"
WHERE ctid IN (
        SELECT ctid
        FROM (
            SELECT ctid, "Coordinates_GZ12_Id", "Coordinates_GZH_Id", "Coordinates_GZC_Id", "Coordinates_GZD_Id", row_number()
            OVER (PARTITION BY "Coordinates_GZ12_Id", "Coordinates_GZH_Id", "Coordinates_GZC_Id", "Coordinates_GZD_Id" ORDER BY ctid) rn
            FROM "FullyCrossMatchedGalaxies"
        ) t
        WHERE rn > 1
        );
```

Create Final dataset:
```sql
SELECT
    "Id",
    "OriginalDataset",
    "OriginalDatasetId",
    "Coordinates_GZ12_Id" AS "GZ12_Id",
    "Coordinates_GZH_Id" AS "GZH_Id",
    "Coordinates_GZC_Id" AS "GZC_Id",
    "Coordinates_GZD_Id" AS "GZD_Id",
    "ra",
    "dec",
    "Coordinates",
    "BinaryClass",
    "z",
    "z_type",
    "DebiasedSpiralFlag",
    "DebiasedSpiralProbability",
    "DebiasedEllipticalFlag",
    "DebiasedEllipticalProbability"
INTO "FinalDataset"
FROM "FullyCrossMatchedGalaxies"
```

# Redshift cleanup

```sql
DELETE
FROM "FinalDataset"
WHERE "z" IS NULL;

DELETE
FROM "FinalDataset"
WHERE "OriginalDataset" IN ('GZ1', 'GZ2') AND ("z" > 0.25 OR "z" < 0.001);

DELETE
FROM "FinalDataset"
WHERE "OriginalDataset" IN ('GZ_DECALS_12', 'GZ_DECALS_5') AND ("z" > 0.15 OR "z" < 0);

DELETE
FROM "FinalDataset"
WHERE "OriginalDataset" = 'GZ_HUBBLE' AND "z" > 4;

DELETE
FROM "FinalDataset"
WHERE "OriginalDataset" = 'GZ_CANDELS' AND "z" > 3;
```